## AWaRe via ATC Codes
This notebook explores an alternative mapping of AWaRe classifications via ATC codes. This is a standalone approach and not directly linked to the dm+d-based mapping used in the main notebook.

### UK Access, Watch, Reserve, and Other classification for antibiotics in dmd

The ["UK Access, Watch, Reserve, and Other classification for antibiotics"](https://www.gov.uk/government/publications/uk-aware-antibiotic-classification/uk-access-watch-reserve-and-other-classification-for-antibiotics-uk-aware-antibiotic-classification) categorises antibiotics into groups such as Access, Watch, and Reserve to guide healthcare professionals in optimising their use and mitigating antimicrobial resistance.

The existing list on the UKHSA site is not machine readable. To facilitate research, it is helpful to link this categorisation to the [dm+d standard](https://www.nhsbsa.nhs.uk/pharmacies-gp-practices-and-appliance-contractors/dictionary-medicines-and-devices-dmd).

### Imports
We need to import some libaries to help with the code

In [1]:
import pandas as pd
import requests
from ebmdatalab import bq
import os

### Getting the AWaRe list
The AWaRe list has been provided by the UKHSA team as an Excel spreadsheet. We can save the Excel spreadsheet as a CSV file then read into Pandas.

In [2]:
excel_path = os.path.join('..', 'data', 'AWaRe_UK_2024_translation_table_FINAL_UPDATED_20250429.xlsx')
df_ukhsa = pd.read_excel(excel_path)
df_ukhsa

,dmd,Antibiotic,Class,atccode,route,VmpUnit,WHO_AWaRe_2023,UK_AWaRe_2024,Include,Notes
0,Amikacin 100mg/2ml solution for injection vials,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN
1,Amikacin 1165mg/100ml solution for injection v...,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN
2,Amikacin 1200mg solution for injection vials,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN
3,Amikacin 1300mg/50ml solution for injection vials,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN
4,Amikacin 250mg/2ml injection,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN
...,...,...,...,...,...,...,...,...,...,...
926,Vancomycin 50mg/5ml oral solution,Vancomycin_oral,Glycopeptides,A07AA09,O,ml,w,w,1,NaN
927,Vancomycin 5mg/0.5ml solution for injection pr...,Vancomycin_IV,Glycopeptides,J01XA01,P,ml,w,w,1,NaN
928,Vancomycin 62.5mg/5ml oral solution,Vancomycin_oral,Glycopeptides,A07AA09,O,ml,w,w,1,NaN
929,Vancomycin 750mg/250ml infusion bags,Vancomycin_IV,Glycopeptides,J01XA01,P,ml,w,w,1,NaN


In [3]:
df_ukhsa_atccodes = df_ukhsa[['atccode', 'Antibiotic', 'WHO_AWaRe_2023', 'UK_AWaRe_2024']].drop_duplicates()
df_ukhsa_atccodes

with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(df_ukhsa_atccodes)

,atccode,Antibiotic,WHO_AWaRe_2023,UK_AWaRe_2024
0,J01GB06,Amikacin,a,w
10,J01CA04,Amoxicillin,a,a
24,J01CA01,Ampicillin,a,a
29,J01FA10,Azithromycin,w,w
35,J01DF51,Aztreonam/avibactam,r,r
36,J01DF01,Aztreonam,r,r
42,J04AK05,Bedaquiline,o,o
43,J01CE08,Benzathine-benzylpenicillin,a,a
49,J01CE01,Benzylpenicillin,a,a
55,J01CE02,Benzylpenicillin,a,a


In [4]:
# Construct the query using the formatted list.
sql = f"""
SELECT DISTINCT
  CAST (vmp.id AS string) AS vmp_id,
  vmp.nm AS vmp_nm,
  vmp.invalid AS invalid_vmp,
  CAST (vtm.id AS string) AS vtm_id,
  vtm.nm AS vtm_nm,
  COALESCE(sroute.route, routelookup.who_route) AS atc_route,
  atclookup.ATC AS atc
FROM `ebmdatalab.dmd.vmp` vmp
LEFT JOIN dmd.ont ont ON vmp.id = ont.vmp
LEFT JOIN dmd.vtm vtm ON vmp.vtm = vtm.id
LEFT JOIN dmd.ontformroute ofr ON ont.form = ofr.cd
LEFT JOIN `dmd.vpi_to_atc_march25` atclookup ON vmp.id = atclookup.VPID
LEFT JOIN chris.vmp_single_route_identifier sroute ON vmp.id = sroute.vmp_id -- this table maps VMP to a specific route where more than one is given
LEFT JOIN `scmd_dmd_views.dmd_to_atc_route` routelookup ON ofr.descr = routelookup.dmd_ofr  -- this table maps dmd route to ATC route
"""

# Define the CSV path for caching the results.
csv_path = os.path.join('..', 'data', 'vmp_vtms.csv')

# Use the cached_read function from the bq library to run the query.
vmp_vtms = bq.cached_read(sql, csv_path=csv_path)

# Force vmp_id and vtm_id to proper string type
vmp_vtms['vmp_id'] = vmp_vtms['vmp_id'].astype('string')
vmp_vtms['vtm_id'] = vmp_vtms['vtm_id'].astype('string')

Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


In [5]:
vmp_vtms

,vmp_id,vmp_nm,invalid_vmp,vtm_id,vtm_nm,atc_route,atc
0,45010811000001105,Lazertinib 80mg tablets,False,44991411000001102,Lazertinib,O,None
1,44117211000001108,Generic DryMax Super dressing 5cm x 5cm square,False,<NA>,None,None,None
2,42801911000001108,Catheter safety valves,False,<NA>,None,None,None
3,41960011000001106,Copper coated barrier dressing sterile with ad...,False,<NA>,None,None,None
4,41535211000001108,Odevixibat 600microgram capsules,False,1179275005,Odevixibat,O,A05AX05
...,...,...,...,...,...,...,...
24698,43907811000001106,Somapacitan 10mg/1.5ml solution for injection ...,False,898161005,Somapacitan,P,H01AC07
24699,43907911000001101,Somapacitan 15mg/1.5ml solution for injection ...,False,898161005,Somapacitan,P,H01AC07
24700,44098211000001102,Budesonide 1mg/2ml nebuliser suspension unit d...,False,774924004,Budesonide,Inhal.solution,R03BA02
24701,43813811000001108,Generic PDE reach5 oral powder 18g sachets,False,<NA>,None,O,n/a


In [7]:
# Merge on ATC codes
df_ukhsa_atccodes_matched = (
    df_ukhsa_atccodes.merge(
        vmp_vtms,
        left_on='atccode',
        right_on='atc',
        how='left'
    )
)

with pd.option_context('display.max_rows', None):
    display(df_ukhsa_atccodes_matched)

,atccode,Antibiotic,WHO_AWaRe_2023,UK_AWaRe_2024,vmp_id,vmp_nm,invalid_vmp,vtm_id,vtm_nm,atc_route,atc
0,J01GB06,Amikacin,a,w,44422511000001103,Amikacin 500mg/2ml solution for injection ampo...,False,774534000,Amikacin,P,J01GB06
1,J01GB06,Amikacin,a,w,44007311000001107,Amikacin 25mg/5ml solution for injection ampoules,False,774534000,Amikacin,None,J01GB06
2,J01GB06,Amikacin,a,w,41823511000001108,Amikacin 500mg/100ml infusion polyethylene bot...,False,774534000,Amikacin,P,J01GB06
3,J01GB06,Amikacin,a,w,39601711000001109,Amikacin liposomal 590mg nebuliser dispersion ...,False,782421008,Amikacin liposomal,Inhal.solution,J01GB06
4,J01GB06,Amikacin,a,w,38244211000001108,Amikacin 400micrograms/0.1ml intravitreal inje...,False,774534000,Amikacin,None,J01GB06
5,J01GB06,Amikacin,a,w,35111911000001104,Amikacin 25mg/5ml solution for injection vials,False,774534000,Amikacin,None,J01GB06
6,J01GB06,Amikacin,a,w,35104511000001105,Amikacin 25mg/5ml solution for injection pre-f...,False,774534000,Amikacin,None,J01GB06
7,J01GB06,Amikacin,a,w,35899911000001109,Amikacin 500mg/2ml solution for injection vials,False,774534000,Amikacin,P,J01GB06
8,J01GB06,Amikacin,a,w,35899811000001104,Amikacin 100mg/2ml solution for injection vials,False,774534000,Amikacin,P,J01GB06
9,J01CA04,Amoxicillin,a,a,34226911000001109,Amoxicillin 500mg/50ml infusion bags,False,774586009,Amoxicillin,P,J01CA04


This gives us a large number of VMPs matched on ATC code. 

Manual review of the VMPs though shows a number of possible issues:
- J01CE02 - Used for some of the benzylpenicillin products but actually corresponds to phenoxymethylpenicillin
- J01CE00  - Used for some of the benzylpenicillin products but doesn't appear to be in use.
- J01DI54 - there is a relevent VMP but this doesn't have an ATC code allocated in the BSA VMP to ATC mapping.
- J01MA13 - used for levofloxacin but actually linked to 	trovafloxacin 
- J04AB02 - used for rifampicin/isoniazid/pyrazinamide combo - but actually relates only to rifampicin - correct code = J04AM05